In [6]:
# === CONFIGURATION ===
HEIGHT = 6
WIDTH = 6
NUM_AGENTS = 2
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09
DISCOUNT = 0.99

CNN_CONV_CHANNELS = [32, 64]
CNN_HEAD_HIDDEN_DIM = 128
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3

# MC Settings
MC_ROLLOUTS = 500      # High samples for accuracy
MC_HORIZON = 500       # Long horizon
SEARCH_STEPS = 1000

PATH_CENTRALIZED = "cen_cnn/model_step9400000.pt"
PATH_DECENTRALIZED_AG0 = "decen_cnn_-12_reward/model_agent0_step16050000.pt"
PATH_DECENTRALIZED_AG1 = "decen_cnn_-12_reward/model_agent1_step16050000.pt"

In [7]:
import torch
import numpy as np
from tqdm import tqdm
import sys
sys.path.append("../") 
from models.value_cnn_new import ValueCNNCentralizedStandard, ValueCNNDecentralizedStandard
from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from tadd_helpers.eval_functions import nearest_policy

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
# Load Models
cent_model = ValueCNNCentralizedStandard(HEIGHT, WIDTH, 0.0, DISCOUNT, CNN_HEAD_HIDDEN_DIM, CNN_HEAD_NUM_LAYERS, CNN_CONV_CHANNELS, CNN_KERNEL_SIZE).to(DEVICE)
cent_model.load_state_dict(torch.load(PATH_CENTRALIZED, map_location=DEVICE))
cent_model.eval()

dec_models = []
for p in [PATH_DECENTRALIZED_AG0, PATH_DECENTRALIZED_AG1]:
    m = ValueCNNDecentralizedStandard(HEIGHT, WIDTH, 0.0, DISCOUNT, CNN_HEAD_HIDDEN_DIM, CNN_HEAD_NUM_LAYERS, CNN_CONV_CHANNELS, CNN_KERNEL_SIZE).to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE))
    m.eval()
    dec_models.append(m)

In [9]:
# =============================================================================
# PHASE 2: MINE INTERESTING STATES (Top 3 Best & Worst)
# =============================================================================

def get_predictions(state):
    # Centralized
    vc = cent_model.get_value(state)
    
    # Decentralized Breakdown
    # Note: We must pass the specific agent position for the decentralized model
    vd0 = dec_models[0].get_value(state, agent_pos=state.agent_position(0))
    vd1 = dec_models[1].get_value(state, agent_pos=state.agent_position(1))
    
    return vc, vd0, vd1

def mine_interesting_states(steps):
    print(f"Generating trajectory of {steps} steps to find Best/Worst disagreements...")
    state = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
    trajectory_data = []

    # We use a progress bar to show scanning
    for _ in tqdm(range(steps)):
        # 1. Get Predictions
        vc, vd0, vd1 = get_predictions(state)
        vd_sum = vd0 + vd1
        diff = vc - vd_sum
        
        # 2. Store Data Snapshot
        trajectory_data.append({
            "state": state.copy(),
            "diff": diff,       # The Disagreement (Cent - Dec)
            "v_cent": vc,
            "v_dec_sum": vd_sum,
            "v_dec_0": vd0,
            "v_dec_1": vd1
        })
        
        # 3. Step Environment (using Nearest Policy to generate valid states)
        idx = state.get_random_agent_id()
        act = nearest_policy(state, idx)
        
        # Physics
        np_pos = np.clip(state.agent_position(idx) + act.vector, [0,0], [HEIGHT-1, WIDTH-1])
        state.set_agent_position(idx, np_pos)
        
        # Apple Logic
        if state.apples[tuple(np_pos)] > 0: 
            state.remove_apple_at(np_pos)
            
        spawn_apples(state, SPAWN_PROB)
        despawn_apples(state, DESPAWN_PROB)
        
    # --- FIND EXTREMES ---
    
    # 1. Worst Disagreements: Largest Positive Difference (Centralized >> Decentralized)
    # We want to see where Centralized is "Right" (28) and Decentralized is "Wrong" (16)
    trajectory_data.sort(key=lambda x: x["diff"], reverse=True)
    top_3_worst = trajectory_data[:3]
    
    # 2. Best Agreements: Smallest Absolute Difference
    # We want to see if there are any states where Decentralized actually works
    trajectory_data.sort(key=lambda x: abs(x["diff"]))
    top_3_best = trajectory_data[:3]
    
    return top_3_worst, top_3_best

# Run the mining to define the variables
worst_states, best_states = mine_interesting_states(SEARCH_STEPS)


# =============================================================================
# PHASE 3: BATCH ANALYSIS (With INDIVIDUAL MC Breakdown)
# =============================================================================

def run_monte_carlo_individual(start_state, rollouts, horizon, desc="MC"):
    """
    Returns (Team_Avg, Agent0_Avg, Agent1_Avg)
    Uses the -1/+2 split for individuals so they sum to the Team Reward (+1).
    """
    returns_team = []
    returns_ag0 = []
    returns_ag1 = []
    
    for _ in tqdm(range(rollouts), desc=desc, leave=False):
        s = start_state.copy()
        
        # Accumulators
        G_team = 0.0
        G_0 = 0.0
        G_1 = 0.0
        
        curr_discount = 1.0
        
        for t in range(horizon):
            # --- 1. HALF-STEP A: Move ---
            active_idx = s.get_random_agent_id()
            action = nearest_policy(s, active_idx)
            new_pos = np.clip(s.agent_position(active_idx) + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
            s.set_agent_position(active_idx, new_pos)
            
            # Reward A: 0 for everyone (Free Walk)
            # Apply Discount 1
            # (No change to G because reward is 0)
            curr_discount *= DISCOUNT
            
            # --- 2. HALF-STEP B: Pick / Spawn ---
            r_team = 0.0
            r_0 = 0.0
            r_1 = 0.0
            
            if s.apples[tuple(new_pos)] > 0:
                s.remove_apple_at(new_pos)
                
                # TEAM Logic: +1 Net
                r_team = 1.0
                
                # INDIVIDUAL Logic: Picker gets -1, Others get +2 (Sum=1)
                # This matches the theoretical distribution the Decen model tried to learn
                if active_idx == 0:
                    r_0 = -1.0
                    r_1 = 2.0
                else:
                    r_0 = 2.0
                    r_1 = -1.0
            
            spawn_apples(s, SPAWN_PROB)
            despawn_apples(s, DESPAWN_PROB)
            
            # Apply Discount 2 & Accumulate
            G_team += curr_discount * r_team
            G_0    += curr_discount * r_0
            G_1    += curr_discount * r_1
            
            curr_discount *= DISCOUNT
            
            if curr_discount < 0.001:
                break
        
        returns_team.append(G_team)
        returns_ag0.append(G_0)
        returns_ag1.append(G_1)
        
    return np.mean(returns_team), np.mean(returns_ag0), np.mean(returns_ag1)


def analyze_batch(state_list, title):
    print(f"\n{'='*20} {title} {'='*20}")
    
    for i, data in enumerate(state_list):
        s = data["state"]
        print(f"\n--- STATE {i+1} ---")
        print(s)
        
        # Run MC with Breakdown
        mc_team, mc_0, mc_1 = run_monte_carlo_individual(s, rollouts=500, horizon=MC_HORIZON, desc=f"State {i+1}")
        
        # Model Predictions
        vc = data["v_cent"]
        vd_sum = data["v_dec_sum"]
        vd0 = data["v_dec_0"]
        vd1 = data["v_dec_1"]
        
        print(f"\nRESULTS:")
        print(f"  TEAM VALUE:")
        print(f"    MC (Ground Truth):   {mc_team:.2f}")
        print(f"    Centralized Pred:    {vc:.2f}   (Error: {vc - mc_team:.2f})")
        print(f"    Decentralized Sum:   {vd_sum:.2f}   (Error: {vd_sum - mc_team:.2f})")
        
        print(f"\n  INDIVIDUAL BREAKDOWN (Decentralized):")
        print(f"    AGENT 0:  Pred: {vd0:.2f}  |  MC Truth: {mc_0:.2f}  |  Diff: {vd0 - mc_0:.2f}")
        print(f"    AGENT 1:  Pred: {vd1:.2f}  |  MC Truth: {mc_1:.2f}  |  Diff: {vd1 - mc_1:.2f}")



Generating trajectory of 1000 steps to find Best/Worst disagreements...


100%|██████████| 1000/1000 [00:11<00:00, 90.85it/s]


In [10]:
# =============================================================================
# PHASE 5: INTENT ANALYSIS (CORRECTED LOGIC)
# =============================================================================

import random 
from orchard.environment import MoveAction  # <--- Added Missing Import

# 1. The EXACT Wrapper from Training (No manual reward hacking)
def get_best_action_decentralized_lookahead(models_list, state, agent_idx):
    """
    Decentralized Lookahead:
    Queries each agent's specific model for the Team Value sum.
    """
    agent_pos = state.agent_position(agent_idx)
    candidates = []
    
    for action in MoveAction:
        new_pos = np.clip(agent_pos + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
        
        s_mid = state.copy()
        s_mid.set_agent_position(agent_idx, new_pos)
        
        # Calculate Team Value (Sum of V_i from Model_i)
        current_team_val = 0.0
        for i in range(NUM_AGENTS):
            pos_i = s_mid.agent_position(i)
            # Query the specific model for agent i
            current_team_val += models_list[i].get_value(s_mid, agent_pos=pos_i)
            
        candidates.append((current_team_val, action))
            
    # --- CRASH PROOFING ---
    valid_candidates = []
    for val, act in candidates:
        if not np.isnan(val) and not np.isinf(val):
            valid_candidates.append((val, act))
            
    if len(valid_candidates) == 0:
        return random.choice(list(MoveAction))
        
    max_val = max(c[0] for c in valid_candidates)
    best_actions = [act for (val, act) in valid_candidates if abs(val - max_val) < 1e-5]
    
    if len(best_actions) == 0:
        return random.choice(list(MoveAction))
    
    return random.choice(best_actions)

# 2. The Opportunity Analysis
def analyze_opportunity_vs_action():
    state = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
    
    # Stats: [Opportunities, Takes]
    # Opportunity = Apple is exactly 1 step away
    # Take = Agent moves onto the apple
    stats = {
        0: {'ops': 0, 'takes': 0},
        1: {'ops': 0, 'takes': 0}
    }
    
    print(f"Analyzing {SEARCH_STEPS} steps for Apple Opportunities...")
    
    for _ in tqdm(range(SEARCH_STEPS)):
        active_idx = state.get_random_agent_id()
        curr_pos = state.agent_position(active_idx)
        
        # A. Check for Opportunity (Is an apple 1 step away?)
        apple_locs = np.argwhere(state.apples > 0)
        is_opportunity = False
        
        if len(apple_locs) > 0:
            # Manhattan distance to all apples
            dists = np.abs(apple_locs - curr_pos).sum(axis=1)
            min_dist = np.min(dists)
            if min_dist == 1:
                is_opportunity = True
        
        # B. Get Action from STRICT Policy
        action = get_best_action_decentralized_lookahead(dec_models, state, active_idx)
        
        # C. Check Decision
        new_pos = np.clip(curr_pos + action.vector, [0,0], [HEIGHT-1, WIDTH-1])
        
        if is_opportunity:
            stats[active_idx]['ops'] += 1
            # Did we move onto an apple?
            if state.apples[tuple(new_pos)] > 0:
                stats[active_idx]['takes'] += 1
        
        # D. Execute Environment Step
        state.set_agent_position(active_idx, new_pos)
        if state.apples[tuple(new_pos)] > 0:
            state.remove_apple_at(new_pos)
            
        spawn_apples(state, SPAWN_PROB)
        despawn_apples(state, DESPAWN_PROB)
        
    # Report
    print(f"\n{'='*60}")
    print(f"NAVIGATION ANALYSIS (When Apple is 1 Step Away)")
    print(f"{'='*60}")
    
    for ag in [0, 1]:
        ops = stats[ag]['ops']
        takes = stats[ag]['takes']
        rate = (takes / ops * 100) if ops > 0 else 0.0
        
        print(f"AGENT {ag}:")
        print(f"  Opportunities: {ops}")
        print(f"  Taken:         {takes}")
        print(f"  Take Rate:     {rate:.1f}%")
        print("-" * 30)

analyze_opportunity_vs_action()

Analyzing 1000 steps for Apple Opportunities...


100%|██████████| 1000/1000 [00:31<00:00, 32.08it/s]


NAVIGATION ANALYSIS (When Apple is 1 Step Away)
AGENT 0:
  Opportunities: 306
  Taken:         120
  Take Rate:     39.2%
------------------------------
AGENT 1:
  Opportunities: 316
  Taken:         280
  Take Rate:     88.6%
------------------------------
